In [ ]:
# 6X6 TANGO/ BINAIRO PUZZLE SOLVER #


import random 
import copy 


def row_populator(row):

  xzero_b = [ ["m","s"], ["s","m"] ]

  ezero_b = [ ["m","m"], ["s","s"] ]

  single_b = { str(["s","b"]): [ ["s","m"], ["s","s"] ], str(["b","s"]): [ ["m","s"], ["s","s"] ], str(["m","b"]): [ ["m","s"],["m","m"] ], str(["b","m"]): [ ["s","m"], ["m","m"] ] }

  double_b = [ ["s","s"], ["m","m"], ["s","m"], ["m","s"] ]

  for element in row: 
    if isinstance(element, list) == True: 
      
      if element.count("b") == 0: 
        if element.count("x") == 2: 
          xchoice = random.choice(xzero_b)
          row[row.index(element)] = xchoice
        if element.count("e") == 2: 
          echoice = random.choice(ezero_b)
          row[row.index(element)] = echoice
      
      if element.count("b") == 1:
        for key in single_b: 
          if str(element) == key:
            sinchoice = random.choice(single_b[key])
            row[row.index(element)] = sinchoice
      
      if element.count("b") == 2: 
        choice = random.choice(double_b)
        row[row.index(element)] = choice
      

  return row


def row_checker(row): 
  inequalities = []
  inequality_score = 0 

  for element in row: 
    if isinstance(element, str) == True and len(element) < 3: 
      if "x" in element: 
        inequalities.append(element)

  for xn in inequalities: 
      if row[row.index(xn) - 1][1] != row[row.index(xn) + 1][0]: 
        inequality_score += 1 

  equalities = []
  equality_score = 0 

  for element in row: 
    if isinstance(element, str) == True and len(element) < 3: 
      if "e" in element: 
        equalities.append(element)

  for en in equalities: 
    if row[row.index(en) - 1][1] == row[row.index(en) + 1][0]: 
      equality_score += 1 
  
  row_string = ""
  for element in row:
    if isinstance(element, list) == True: 
      for letter in element: 
        row_string = row_string + letter

  
  
  if len(inequalities) == inequality_score and len(equalities) == equality_score and row_string.count("m") == 3 and row_string.count("s") == 3 and "mmm" not in row_string and "sss" not in row_string: 
    return "Good"
  else: 
    return "Bad"
  

def brute_force(row, row_preserved): 
  
  while row_checker(row) != "Good": 
    row = row_preserved[:]
    row_populator(row)
    row_checker(row)
  

  row_final = []
  for element in row: 
    if isinstance(element, list) == True: 
      row_final.append(element)
  
  return row_final 

# ^^^ (CONSTRAINED ROW) GENERATION MECHANICS ^^^
##################################################



def grid_populator(grid): 

  permutations = { "p0": [ ["s","s"],["m","s"],["m","m"] ], "p1": [ ["s","s"],["m","m"],["s","m"] ], "p2": [ ["m","m"],["s","s"],["m","s"] ], "p3": [ ["m","m"],["s","m"],["s","s"] ], "p4": [ ["s","m"],["m","s"],["m","s"] ], "p5": [ ["s","m"],["m","s"],["s","m"] ], "p6": [ ["s","m"],["s","s"],["m","m"] ], "p7": [ ["s","m"],["s","m"],["s","m"] ], "p8": [ ["s","m"],["s","m"],["m","s"] ], "p9": [ ["m","s"],["m","s"],["m","s"] ], "p10": [ ["m","s"],["m","s"],["s","m"] ], "p11": [ ["m","s"],["m","m"],["s","s"] ], "p12": [ ["m","s"],["s","m"],["s","m"] ], "p13": [ ["m","s"],["s","m"],["m","s"] ] }

  for row in grid: 

    b_count = 0 
    for element in row: 
      b_count += element.count("b")


    if "x" not in row and "e" not in row and b_count == 6 and len(row) == 3: 
      pchoice = random.choice(list(permutations.values()))
      
      while grid.count(pchoice) >= 4: 
        pchoice = random.choice(list(permutations.values()))
      grid[grid.index(row)] = pchoice
      

    if "x" in row or "e" in row or b_count != 6 or len(row) != 3:
      row_preserved = row[:]
      row_populator(row)
      row_checker(row)
      brute_force(row, row_preserved)
      grid[grid.index(row)] = brute_force(row, row_preserved)  
  
  return grid


def grid_row_checker(grid): 

  gr_penalty = 0 

  new_grid = [] 
  index = -1  
  for row in grid: 
    row_original = row[:]
    index += 1 
    row.append(index)
    new_grid.append(row)
    grid[grid.index(row)] = row_original
  

  checked = []
  for chosen_row in grid: 
    if grid.count(chosen_row) == 3 and chosen_row not in checked: 

      instances = []
      for general_row in new_grid: 
        if general_row[ : len(general_row) - 1] == chosen_row: 
          instances.append(general_row)

      if abs(new_grid.index(instances[2]) - new_grid.index(instances[1])) == abs(new_grid.index(instances[1]) - new_grid.index(instances[0])): 
        gr_penalty += 1 

  return gr_penalty    
     

def column_checker(grid,column_constraints): 


  cc_penalty = 0 

  equalities = [ ["m","m"],["s","s"] ]
  inequalities = [ ["m","s"],["s","m"] ]

  collection = [] 
  for row in grid:
    for pair in row: 
      for element in pair: 
        collection.append(element)

  columns = []
  for j in range(0,6,1):
    column_raw = []
    for i in range(j,36,12):
      pair = collection[i] + " " + collection[i + 6]
      column_raw.append(pair.split(" "))
    columns.append(column_raw)


  for col in column_constraints: 
    cc_pairs_only = [] 
    for element in col: 
      if isinstance(element,list) == True:
        cc_pairs_only.append(element)
    # at this point, you have only pair lists for one column
    for pair in cc_pairs_only:
      if pair.count("b") == 0: 

        if pair == ["x","x"] and columns[column_constraints.index(col)][cc_pairs_only.index(pair)] not in inequalities:
          cc_penalty += 1 

        if pair == ["e","e"] and columns[column_constraints.index(col)][cc_pairs_only.index(pair)] not in equalities: 
          cc_penalty += 1 


  for j in range(0,6):  
    cc_preserved = column_constraints[j][:]
    for i in range(0,3): 
      add_count = 0 
      for k in range(0,len(column_constraints[j])): 
        if add_count < 1:
          if isinstance(column_constraints[j][k],list) == True and column_constraints[j][k].count("m") == 0 and column_constraints[j][k].count("s") == 0:
            column_constraints[j][k] = columns[j][i]
            add_count += 1
    # At this point, we have a constraint column modified to have the populated pair list and not the constraint pair list.
    if row_checker(column_constraints[j]) == "Bad":
      cc_penalty += 1 
    column_constraints[j] = cc_preserved


  return cc_penalty 

# ^^^ (CONSTRAINED GRID) GENERATION MECHANICS ^^^
###################################################


grid = [ [ ["b","s"], ["s","b"], ["b","b"] ], [ ["b","b"],["s","b"],"e1",["b","b"] ], [ ["b","s"],["m","b"],["b","b"] ], [ ["b","s"],["b","b"],"e1",["b","b"] ], [ ["b","m"],["s","b"],["b","b"] ], [ ["b","b"],["b","b"],"e1",["b","b"] ] ] # fill here

grid_preserved = copy.deepcopy(grid)

column_constraints = [ [ ["b","b"],["b","b"],["b","b"] ],[ ["b","b"],["b","b"],["b","b"] ],[ ["b","b"],["b","b"],["b","b"] ],[ ["b","b"],["b","b"],"x1",["x","x"] ],[ ["b","b"],"e1",["x","x"],["b","b"] ],[ ["b","b"],["b","b"],["b","b"] ] ] # fill here 


def brute_force_2(grid,grid_preserved,column_constraints): 

 grid_populator(grid)
 grid_row_checker(grid)
 column_checker(grid,column_constraints)
 

 while grid_row_checker(grid) != 0 or column_checker(grid,column_constraints) != 0: 
   grid = copy.deepcopy(grid_preserved)
   grid_populator(grid)
   grid_row_checker(grid)
   column_checker(grid,column_constraints)

 print("Solved Grid:","\n")
 for row in grid: 
   print("Row " + str(grid.index(row) + 1) + ":",row,"\n")

print(brute_force_2(grid,grid_preserved,column_constraints))